# Linear vs Polynomial Regression on Curved Data

**Goal:** Show what happens when a straight line is fitted to data that follows a curve, and how polynomial features fix this.

**Workflow:**
1. Import libraries
2. Generate a synthetic (made-up) dataset with a curved relationship
3. Look at the data
4. Split into training and test sets
5. Fit a plain **Linear Regression** and measure R²
6. Add **polynomial features** and fit again, then compare R²

## 1. Import Libraries

- **pandas** (`pd`): tabular data handling. Imported here but not used in this notebook.
- **numpy** (`np`): arrays and random number generation. Used to create the dataset.
- **seaborn** (`sns`): statistical plotting. Imported but not used here.
- **matplotlib.pyplot** (`plt`): used to draw the scatter plot.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## 2. Create a Synthetic Dataset

We build our own data so we know the true relationship in advance.

- `np.random.rand(100, 1)` draws 100 numbers uniformly between 0 and 1. Multiplying by 6 and subtracting 3 spreads them over **[-3, 3)**. The shape `(100, 1)` makes `X` a column of 100 rows, which is the format scikit-learn expects.
- The target is generated from this formula:

  $$y = 0.5\,x^2 + 1.5\,x + \varepsilon$$

  The first two terms form a **parabola** (a curve). `np.random.randn(100, 1)` adds **random noise** from a standard normal distribution, so the points scatter around the curve, as real data does.

**Note:** no random seed is set, so every run creates different numbers. The printed values below will differ from run to run.

In [ ]:
# Creating our own dataset
X = 6 * np.random.rand(100, 1) - 3
y = 0.5 * X**2 + 1.5 * X + np.random.randn(100, 1)

## 3. Inspect the Features (X)

Displaying `X` shows the 100 input values, one per row. Each value is a single feature, so the model has only one input. Values are mostly between -3 and 3.

In [ ]:
X

## 4. Inspect the Target (y)

`y` holds the output for each `X`. Values grow quickly at both ends because of the x² term, and the random noise makes each value slightly different from the exact curve.

In [ ]:
y

## 5. Plot the Data

`plt.scatter` plots each `(X, y)` pair as a green point. The points form a **U-shaped curve**, not a straight line. This matters: a straight-line model cannot follow this shape, which is the point of the next steps.

We add axis labels so the plot is easier to read.

In [ ]:
plt.scatter(X, y, color='g')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Synthetic data with a curved relationship')
plt.show()

## 6. Train-Test Split

`train_test_split` puts 75% of the rows into the **training set**, which the model learns from, and 25% into the **test set**, which it never sees during training. The test set tells us how well the model works on new data.

`random_state=42` makes the shuffle repeatable, so the same rows always land in the same set.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

## 7. Simple Linear Regression

`LinearRegression` fits a straight line:

$$\hat{y} = b_0 + b_1 x$$

`.fit()` finds the slope $b_1$ and intercept $b_0$ that minimize the squared errors on the training data. Because the true relationship is curved, we expect this straight line to miss the pattern.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

## 8. Predict on the Test Set

`.predict()` applies the fitted line to the test inputs and returns one predicted `y` per test row.

In [ ]:
y_pred = model.predict(X_test)

## 9. Evaluate with R²

**R²** (coefficient of determination) is the share of the variation in `y` that the model explains.
- **1.0**: perfect predictions.
- **0**: no better than always predicting the average of `y`.
- **Below 0**: worse than predicting the average.

`r2_score(y_test, y_pred)` compares the true test values with the predictions.

In the original run, the linear model scored about **0.63**. That is decent, but it is limited by the straight line, which cannot bend to follow the curve. Your value will differ because the data is random.

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
r2 = r2_score(y_test, y_pred)
r2

## 10. Polynomial Features

`PolynomialFeatures(degree=2, include_bias=True)` expands the single input `x` into several columns:

$$x \rightarrow [\,1,\; x,\; x^2\,]$$

- `1` is the bias column (a constant of 1). `include_bias=True` adds it. `LinearRegression` already has an intercept, so this column is redundant, but it does no harm.
- `x²` is the new feature that lets the model capture curvature.

The model is still a **linear** regression. It is linear in the *new* columns, which is why this is called polynomial regression. The learned formula becomes $\hat{y} = b_0 + b_1 x + b_2 x^2$.

We call `fit_transform` on the training data, which learns the expansion, and only `transform` on the test data. Fitting on the test data would be a form of data leakage.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=True)

In [ ]:
X_train_poly = poly.fit_transform(X_train)   # learn and apply the expansion on training data

In [ ]:
X_test_poly = poly.transform(X_test)         # apply the same expansion to test data

## 11. Fit the Polynomial Model

We train a new `LinearRegression` on the expanded features. The code is identical to step 7. Only the inputs have changed.

In [ ]:
model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train)

## 12. Predict and Evaluate the Polynomial Model

We predict on the expanded test features and compute R² the same way as before, so the two results can be compared directly.

In the original run, the polynomial model scored about **0.85**, up from about 0.63. The improvement comes from the curved term matching the true relationship (the data was generated with an x² term).

In [ ]:
y_pred_poly = model_poly.predict(X_test_poly)

In [ ]:
r2_poly = r2_score(y_test, y_pred_poly)
r2_poly

## 13. Compare the Two Models

Printing both scores side by side makes the difference clear. Higher R² is better.

In [ ]:
print(f'Linear Regression R²:      {r2:.3f}')
print(f'Polynomial Regression R²:  {r2_poly:.3f}')

## 14. Summary

- The data follows a **curve**, so a straight line (`degree=1`) can only partly explain it.
- Adding an **x² feature** (`degree=2`) lets the same linear-regression machinery fit the curve. R² rises substantially.
- Degree 2 works best here because the true formula is quadratic. In real projects, the right degree is unknown.

**Cautions and next steps:**
- **Overfitting:** a very high degree (e.g. 10 or 15) can follow the noise and score well on training data but poorly on new data. Try degrees 1 to 6 and compare test scores.
- **Small test set:** 25 points means the R² estimate is noisy. Use `cross_val_score` (as in the previous notebook) for a more stable comparison.
- **Pipelines:** `sklearn.pipeline.make_pipeline(PolynomialFeatures(2), LinearRegression())` combines the expansion and the model into one object, which reduces the chance of applying steps in the wrong order.